In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
data = np.load("go_chunks/chunk_0000.npz")
data

NpzFile 'go_chunks/chunk_0000.npz' with keys: positions, next_to_go, winner

In [2]:
def load_data(path):
    with np.load(path) as data:
        data_positions = data["positions"]
        data_labels = data["winner"]

        

    data_positions = np.transpose(data_positions, (0, 2, 3, 1)) 
    return data_positions, data_labels


train_positions, train_labels = load_data("go_chunks/chunk_0000.npz")
test_positions, test_labels = load_data("go_chunks/chunk_0001.npz")


In [3]:
import os 

cuttoff=-1

def load_data_tf(paths):
    for path in paths:
        with np.load(path) as data:
            data_positions = data["positions"]
            data_labels = data["winner"]

        data_positions = np.transpose(data_positions, (0, 2, 3, 1)) 
        yield data_positions, data_labels

paths = [f"go_chunks/{f}" for f in os.listdir("go_chunks") if f.endswith(".npz")][:cuttoff]
#paths= ["go_chunks/chunk_0000.npz","go_chunks/chunk_0002.npz","go_chunks/chunk_0003.npz"]
ds = tf.data.Dataset.from_generator(load_data_tf, args=(paths,),output_signature=(
        tf.TensorSpec(shape=(None, 19, 19, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.int64)
    ))

#ds = ds.flat_map(lambda x, y: tf.data.Dataset.from_tensor_slices((x, y)))


In [14]:
import os 

cuttoff=-1
paths = [f"go_chunks/{f}" for f in os.listdir("go_chunks") if f.endswith(".npz")][:cuttoff]

def load_data_tf(paths):
    for path in paths:
        with np.load(path) as data:
            data_positions = data["positions"]
            data_labels = data["winner"]
        data_positions = np.transpose(data_positions, (0, 2, 3, 1))
        for pos, label in zip(data_positions, data_labels):
            yield pos, label  # (19, 19, 3) and scalar

ds = tf.data.Dataset.from_generator(
    load_data_tf,
    args=(paths,),
    output_signature=(
        tf.TensorSpec(shape=(19, 19, 3), dtype=tf.float32),  # single sample
        tf.TensorSpec(shape=(), dtype=tf.int64)              # scalar label
    )
)
ds = ds.shuffle(10000).batch(256).prefetch(tf.data.AUTOTUNE).repeat()
# No flat_map — just shuffle, batch, prefetch
#ds = ds.shuffle(10000).batch(256).prefetch(tf.data.AUTOTUNE).repeat()

In [4]:
import tensorflow as tf

#train_dataset = tf.data.Dataset.from_tensor_slices((train_positions, train_labels))
test_dataset = tf.data.Dataset.from_tensor_slices((test_positions, test_labels))

BATCH_SIZE = 254
SHUFFLE_BUFFER_SIZE = 100

#train_dataset = train_dataset.shuffle(SHUFFLE_BUFFER_SIZE).batch(BATCH_SIZE).repeat()
test_dataset = test_dataset.batch(BATCH_SIZE)

In [ ]:
inputs = layers.Input(shape=(19, 19, 3))

# --- All CNN blocks ---
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)  # (B, 4, 4, 64)

# --- Self-attention after all CNNs ---
_, H, W, C = x.shape
x = layers.Reshape((H * W, C))(x)                                     # (B, 16, 64)
x = layers.MultiHeadAttention(num_heads=2, key_dim=32)(x, x)
x = layers.Reshape((H, W, C))(x)

x = layers.Flatten()(x)
outputs = layers.Dense(1, activation='relu')(x)

model = models.Model(inputs=inputs, outputs=outputs)



In [15]:
from tensorflow.keras import layers
from tensorflow.keras import models
inputs = layers.Input(shape=(19, 19, 3))

# Block 1
x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)

# Block 2
x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)

# Block 3
x = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
x = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)

x = layers.Flatten()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.01)(x)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(1)(x)

model = models.Model(inputs=inputs, outputs=outputs)

In [ ]:

model.summary()
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'])
history = model.fit(ds, epochs=10, batch_size=250,validation_data=test_dataset)

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 19, 19, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 19, 19, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 19, 19, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 19, 19, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 19, 19, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_24 (Conv2D)              │ (None, 19, 19, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 19, 19, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_25 (Conv2D)              │ (None, 19, 19, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_26 (Conv2D)              │ (None, 19, 19, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 19, 19, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 92416)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │    23,658,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,822,465 (94.69 MB)

 Trainable params: 24,821,569 (94.69 MB)

 Non-trainable params: 896 (3.50 KB)

Epoch 1/10
    259/Unknown 314s 1s/step - accuracy: 0.8298 - loss: -176.3904

In [10]:
model.save_weights('./checkpoints/Weird_inverse_funnel_084acc.weights.h5')